# Lab 05: Methods (The Mirage)
Prompting `gemma3:1b` via Ollama to generate a JSON config. We strictly enforce the schema.


In [ ]:
import requests
import json
import re

def ask_llm(prompt):
    try:
        res = requests.post("http://localhost:11434/api/generate", 
                            json={"model": "gemma3:1b", "prompt": prompt, "stream": False})
        res.raise_for_status()
        return res.json()['response']
    except Exception as e:
        return str(e)

prompt = "Propose a systolic array configuration for an XR SoC. Output ONLY valid JSON with keys 'ArrayHeight' and 'ArrayWidth' (integers between 8 and 128)."
raw_resp = ask_llm(prompt)
print("Raw AI Output:", raw_resp)

# Extract JSON using regex
match = re.search(r'\{.*\}', raw_resp.replace('\n', ''))
if match:
    config = json.loads(match.group(0))
else:
    print("AI hallucinated formatting. Using fallback.")
    config = {"ArrayHeight": 8, "ArrayWidth": 8}

with open('.arch2_state.json', 'r') as f: state = json.load(f)
state['ai_config'] = config
with open('.arch2_state.json', 'w') as f: json.dump(state, f)
print("Config saved to state:", config)
